# Exercise 1
Build a Polite News Crawler: Implement a domain-specific scraper that collects
news articles about Artificial Intelligence. As the base of your crawler use BeautifulSoup.

**Tasks**  
• Crawl at least 3 different news websites.  
• Collect at least 200 articles and 100,000 words total.  
• Store URL, title, publication date, article text, domain, and crawl timestamp.  
• Respect robots.txt.  
• Implement rate limiting (≥ 1 second delay).  
• Avoid duplicate URLs.  

## Imports

In [11]:
import time
import json
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from urllib.robotparser import RobotFileParser
from datetime import datetime, UTC

## fetch HTML with rate limiting
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to fetch htmls and use the following for it: r = requests.get(url, headers, timeout=10)

In [12]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; PoliteNewsCrawler/1.0; +https://example.com/bot-info)"
}
RATE_LIMIT = 1.0
OUTPUT_FILE = "ai_news.jsonl"

MAX_ARTICLES = 200
MIN_WORDS = 100_000

AI_KEYWORDS = [
    "ai", "artificial intelligence",
    "machine learning", "neural network",
    "chatgpt", "openai", "deep learning"
]

def ai_related(text):
    text = text.lower()
    return any(k in text for k in AI_KEYWORDS)

In [ ]:
visited_urls = set()
robot_cache = {}
total_articles = 0
total_words = 0

## fetch HTML with rate limiting
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to fetch htmls and use the following for it: r = requests.get(url, headers, timeout=10)

In [ ]:
def fetch_html(url):
    time.sleep(RATE_LIMIT)
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        r.raise_for_status()
        return r.text
    except:
        print("Fetch failed:", url)
        return None

## Check robots.txt
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to check the robot txt and use the following for it: rp = RobotFileParser()    

In [ ]:
def robots_allowed(url, user_agent="*"):
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    if base not in robot_cache:
        rp = RobotFileParser()
        rp.set_url(f"{base}/robots.txt")
        try:
            rp.read()
        except:
            return False
        robot_cache[base] = rp

    return robot_cache[base].can_fetch(user_agent, url)

## JSON

In [ ]:
def write_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

## Filter only AI related articles

In [ ]:
def ai_related(text):
    text = text.lower()
    return any(k in text for k in AI_KEYWORDS)

## Extraction from sides

#### Get links
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to get the link from sitemaps

In [ ]:
def get_urls_from_sitemap(sitemap_url, must_contain=None, limit=None):
    html = fetch_html(sitemap_url)
    if not html:
        return []

    root = ET.fromstring(html)
    urls = []
    ns = "{http://www.sitemaps.org/schemas/sitemap/0.9}"

    if root.tag.endswith("sitemapindex"):
        for sm in root.findall(f"{ns}sitemap"):
            loc = sm.find(f"{ns}loc")
            if loc is not None:
                sub = get_urls_from_sitemap(loc.text, must_contain, limit)
                urls.extend(sub)
                if limit and len(urls) >= limit:
                    return urls[:limit]
    elif root.tag.endswith("urlset"):
        for url_el in root.findall(f"{ns}url"):
            loc = url_el.find(f"{ns}loc")
            if loc is not None:
                url = loc.text.strip()
                if must_contain:
                    if not any(x in url for x in must_contain):
                        continue
                urls.append(url)
                if limit and len(urls) >= limit:
                    return urls
    return urls

# some sites don't have sitemaps but do have RSS feeds, so I tried to extract URLs from those
def get_urls_from_rss(feed_url, limit=None):
    html = fetch_html(feed_url)
    if not html:
        return []

    root = ET.fromstring(html)
    urls = []
    for item in root.findall(".//item"):
        link = item.find("link")
        if link is not None and link.text:
            urls.append(link.text.strip())
        if limit and len(urls) >= limit:
            break
    return urls

### Website Ars Technica

In [ ]:
def extract_article_arstechnica(html, url):
    soup = BeautifulSoup(html, "lxml")
    article = soup.select_one("article")
    if not article:
        return None

    title = article.select_one("h1")
    date = article.select_one("time")
    body = article.select("div.post-content p")
    if not title or not body:
        return None

    text = " ".join(p.get_text(" ", strip=True) for p in body)
    return {
        "url": url,
        "title": title.get_text(strip=True),
        "date": date.get("datetime") if date else None,
        "text": text,
        "domain": urlparse(url).netloc,
        "crawl_timestamp": datetime.now(UTC).isoformat()
    }

### Website Techcrunch

In [ ]:
def extract_article_techcrunch(html, url):
    soup = BeautifulSoup(html, "lxml")
    article = soup.select_one("article")
    if not article:
        return None

    title = article.select_one("h1")
    date = article.select_one("time")
    body = article.select("div.entry-content p")
    if not title or not body:
        return None

    text = " ".join(p.get_text(" ", strip=True) for p in body)
    return {
        "url": url,
        "title": title.get_text(strip=True),
        "date": date.get("datetime") if date else None,
        "text": text,
        "domain": urlparse(url).netloc,
        "crawl_timestamp": datetime.now(UTC).isoformat()
    }

### Website Wired

In [ ]:
def extract_article_wired(html, url):
    soup = BeautifulSoup(html, "lxml")
    title = soup.select_one("h1")
    body = soup.select("div[class*='article__body'] p")
    date = soup.select_one("time")
    if not title or not body:
        return None

    text = " ".join(p.get_text(" ", strip=True) for p in body)
    return {
        "url": url,
        "title": title.get_text(strip=True),
        "date": date.get("datetime") if date else None,
        "text": text,
        "domain": urlparse(url).netloc,
        "crawl_timestamp": datetime.now(UTC).isoformat()
    }

## crawl websites 
**AI Use:** Microsoft Copilot GPT-5.1  
*Prompt:* write me a short function to crawl a website

In [17]:
def crawl_urls(urls, extractor):
    global total_articles, total_words
    for url in urls:
        if url in visited_urls:
            continue
        visited_urls.add(url)

        if not robots_allowed(url):
            continue

        html = fetch_html(url)
        if not html:
            continue

        article = extractor(html, url)
        if not article:
            continue

        if not ai_related(article["title"] + " " + article["text"]):
            continue

        words = len(article["text"].split())
        total_words += words
        total_articles += 1

        write_jsonl(OUTPUT_FILE, article)
        print(f"[{total_articles}] {article['domain']} ({words} words)")

        if total_articles >= MAX_ARTICLES and total_words >= MIN_WORDS:
            return

### get list of urls from each website

In [22]:
ars_urls = get_urls_from_sitemap(
    "https://arstechnica.com/sitemap.xml",
    must_contain=["/science/", "/information-technology/"],
    limit=250
)

tc_urls = get_urls_from_rss(
    "https://techcrunch.com/tag/artificial-intelligence/feed/",
    limit=200
)

wired_urls = get_urls_from_rss(
    "https://www.wired.com/feed/category/artificial-intelligence/rss",
    limit=200
)  # Wired AI RSS feed :contentReference[oaicite:2]{index=2}

print(len(ars_urls), len(tc_urls), len(wired_urls))

250 20 20


In [23]:
crawl_urls(ars_urls, extract_article_arstechnica)
crawl_urls(tc_urls, extract_article_techcrunch)
crawl_urls(wired_urls, extract_article_wired)

print("DONE")
print("Articles:", total_articles)
print("Words:", total_words)

[179] arstechnica.com (149 words)
[180] arstechnica.com (167 words)
[181] arstechnica.com (935 words)
[182] arstechnica.com (92 words)
[183] arstechnica.com (74 words)
[184] arstechnica.com (123 words)
[185] arstechnica.com (106 words)
[186] arstechnica.com (118 words)
[187] arstechnica.com (182 words)
[188] arstechnica.com (115 words)
[189] arstechnica.com (1032 words)
[190] arstechnica.com (199 words)
[191] arstechnica.com (129 words)
[192] arstechnica.com (234 words)
[193] arstechnica.com (219 words)
[194] arstechnica.com (134 words)
[195] arstechnica.com (174 words)
[196] arstechnica.com (145 words)
[197] arstechnica.com (170 words)
[198] arstechnica.com (1194 words)
[199] arstechnica.com (170 words)
[200] arstechnica.com (181 words)
DONE
Articles: 200
Words: 184384


## Reflection

**AI Use:** Chat GPT  
*Prompt:* refine the following notes in a more professional way: ### ethical considerations  
  
Downloading the articles without the authors consent for our own usecase might not be in the authors intentions with the article. If we use it in further tasks for things that the author does not agree on he has no chance to do anything against it, as he/she has no notice that his content was crawled and will be used. Another problem is the value of the possible product/usecase that the data collected will be used for. The authors put in their own work and wil not receive any money gained by the finished products. The robots.txt was checked to make sure that only allowed scraping was practiced. ### challenges
Finding the right code lines for the body = soup.select("div[class*='article__body'] p") was quite difficult as this depends on the websites html design. It was not possible to retrieve all the urls from the sitemaps of the websites, therefor I had to extrext them from the rss. ### data quality issues
not every website has the same format in terms of the datetime and so on, also the text might not be complete or consistent for all of the scraped websites


### Ethical Considerations

Scraping and downloading articles for analysis raises several ethical considerations. The authors of the articles did not explicitly consent to their content being collected and used for external purposes such as data analysis or downstream applications. Consequently, the use of this content may not align with the authors’ original intentions. Furthermore, if the scraped data were used in future projects or products with which the authors disagree, they would have no practical way to object, as they are not notified that their content has been collected or reused. Another concern relates to potential economic value: authors invest time and effort into producing the original content but would not receive compensation if the collected data were used to develop valuable products or services. To address responsible data collection practices, the robots.txt files of the respective websites were checked to ensure that only permitted scraping activities were conducted.

### Challenges

One of the main challenges was identifying the correct HTML elements that contain the main article content. For example, extracting the body text using soup.select("div[class*='article__body'] p") required careful inspection of the website’s HTML structure, which varies across different sources. Because the extraction logic depends heavily on the specific design of each website, determining the correct selectors was time-consuming. Another challenge was that not all article URLs could be retrieved directly from the websites’ sitemaps. As a result, additional URLs had to be collected from RSS feeds to ensure sufficient coverage of relevant articles.

### Data Quality Issues

Several factors may affect the quality and consistency of the collected dataset. Different websites use varying formats for metadata such as publication dates and timestamps, which can complicate standardized data processing. In addition, the extracted article text may not always be complete or consistent across all sources due to differences in HTML structure or limitations in the extraction process.